# Notebook 07: Decision Intelligence Recommendations

## Project
NEM Network Constraints & Price Divergence Intelligence System

## Business Question
How can classified price spike and network signals be translated into practical recommendations for energy traders, analysts, and market operators?

## Purpose Of This Notebook
Notebook 07 converts analytical outputs into decision intelligence.

In Notebook 06, each price spike interval was classified by likely driver:

- Constraint-driven
- Interconnector/congestion-driven
- Demand-driven
- Volatility-driven
- Unknown

This notebook turns those classifications into business-facing interpretation:

- Market Situation
- Insight
- Recommendation
- Risk
- Confidence

The goal is to make the analysis useful for decision-making, not just technical reporting.


## How This Notebook Links To The Full Project

Project 2 is designed as a professional energy market analyst portfolio project.

The technical workflow has built the analytical evidence:

1. Price and demand data.
2. Base market features.
3. Constraint activity.
4. Interconnector flow behaviour.
5. Regional price divergence.
6. Spike classification.

Notebook 07 creates the commercial interpretation layer.

This is the point where the project becomes decision-oriented. Instead of only showing data, the analysis explains what market participants should watch, what risks may be emerging, and how confident the recommendation is.


## Decision Intelligence Framework

Each recommendation will contain:

| Field | Meaning |
|---|---|
| `market_situation` | Short description of the market condition observed. |
| `insight` | Analyst interpretation of what the condition means. |
| `recommendation` | Practical action or monitoring recommendation. |
| `risk` | Main risk associated with the condition. |
| `confidence` | Confidence level based on the classification and supporting signals. |

### Important Analytical Note
These recommendations are decision-support outputs. They are not trading instructions. They help an analyst or trader prioritise which intervals and conditions require closer review.


In [3]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 160)


## Step 2: Project Paths And Input Files

### What we are doing
We are defining the project folder, output folder, and input files from Notebook 06.

### Why it matters
The decision intelligence layer depends on the spike classification outputs created in the previous notebook.


In [5]:
PROJECT_ROOT = (
    Path.cwd().parents[0]
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"

SPIKE_EXPLANATION_FILE = OUTPUT_DIR / "06_network_spike_explanation.csv"
MARKET_CLASSIFICATION_FILE = OUTPUT_DIR / "06_market_features_with_spike_classification.csv"

print("Project root:", PROJECT_ROOT)
print("Spike explanation input:", SPIKE_EXPLANATION_FILE)
print("Market classification input:", MARKET_CLASSIFICATION_FILE)


Project root: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence
Spike explanation input: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/06_network_spike_explanation.csv
Market classification input: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/06_market_features_with_spike_classification.csv


## Step 3: Load Spike Explanation Data

### What we are doing
We are loading the spike-level classification output from Notebook 06.

### Why it matters
This table contains the high-price events and their likely driver classification. It is the main input for decision recommendations.


In [6]:
spike_explanation = pd.read_csv(
    SPIKE_EXPLANATION_FILE,
    parse_dates=["settlementdate"]
)

print("Spike explanation shape:", spike_explanation.shape)

spike_explanation.head()


Spike explanation shape: (53, 22)


,settlementdate,regionid,rrp,spike_driver_classification,classification_confidence,driver_signal_count,demand_tightness_signal,volatility_signal,constraint_signal,interconnector_congestion_signal,totaldemand,availablegeneration,supply_demand_gap,price_change,rolling_rrp_volatility_1h,active_constraint_count,max_constraint_marginal_value,congestion_signal_count,vic_nsw_congestion_signal_flag,absolute_price_spread,divergence_flag,extreme_divergence_flag
0,2026-02-05 14:30:00,NSW1,20300.00000,Interconnector/congestion-driven,High,3,True,True,False,True,10190.15,15492.52355,5302.37355,19842.65742,5793.809073,24,12.12,4.0,True,20308.00000,True,True
1,2026-02-06 14:05:00,NSW1,19036.53000,Interconnector/congestion-driven,High,2,False,True,False,True,9651.93,15075.06206,5423.13206,18882.33000,5801.771238,23,5.34,2.0,False,19022.81000,True,True
2,2026-02-05 18:05:00,NSW1,9914.02372,Constraint-driven,High,4,True,True,True,True,11811.76,13709.77609,1898.01609,9634.29090,3719.949900,22,20.23,2.0,False,9794.69790,True,True
3,2026-02-05 17:55:00,NSW1,9911.42681,Interconnector/congestion-driven,High,3,True,True,False,True,11901.45,14019.02209,2117.57209,9608.77893,2762.209380,24,4.96,2.0,False,9854.92681,True,True
4,2026-02-05 16:35:00,NSW1,9240.83000,Constraint-driven,High,4,True,True,True,True,11896.59,14384.92803,2488.33803,8641.47000,2519.281003,29,4.34,2.0,False,9231.88000,True,True


## Step 4: Load Full Market Classification Dataset

### What we are doing
We are loading the full interval-level dataset with spike classifications.

### Why it matters
This table provides wider context for recommendations and summary reporting.


In [7]:
market_classification = pd.read_csv(
    MARKET_CLASSIFICATION_FILE,
    parse_dates=["settlementdate"]
)

print("Market classification shape:", market_classification.shape)

market_classification.head()


Market classification shape: (16126, 56)


,settlementdate,regionid,rrp,intervention,totaldemand,availablegeneration,netinterchange,intervention_regionsum,supply_demand_gap,price_spike_flag,price_change,rolling_rrp_volatility_1h,date,hour,weekday,is_weekend,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree,total_absolute_flow,max_absolute_flow,average_absolute_flow,max_absolute_flow_change,high_flow_interconnector_count,high_flow_change_interconnector_count,congestion_signal_count,congestion_signal_flag,vic_nsw_flow,vic_nsw_absolute_flow,vic_nsw_flow_change,vic_nsw_absolute_flow_change,vic_nsw_high_flow_flag,vic_nsw_high_flow_change_flag,vic_nsw_congestion_signal_flag,nsw_rrp,vic_rrp,price_spread,absolute_price_spread,divergence_flag,extreme_divergence_flag,divergence_direction,high_demand_threshold,low_supply_demand_gap_threshold,high_volatility_threshold,high_price_change_threshold,high_constraint_count_threshold,high_constraint_mv_threshold,demand_tightness_signal,volatility_signal,constraint_signal,interconnector_congestion_signal,spike_driver_classification,driver_signal_count,classification_confidence
0,2026-02-01 00:05:00,NSW1,57.05984,0,8186.33,13272.53373,-1054.73,0,5086.20373,False,NaN,NaN,2026-02-01,0,Sunday,True,17,135.3015,-902.14975,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,57.05984,47.06047,9.99937,9.99937,False,False,NSW1 higher than VIC1,9834.07,3891.103892,20.448633,20.99,25.0,15.37,False,False,True,False,Not a spike,1,Not applicable
1,2026-02-01 00:10:00,NSW1,64.51979,0,8165.29,13246.75420,-924.67,0,5081.46420,False,7.45995,NaN,2026-02-01,0,Sunday,True,16,40.2794,-1006.87256,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,64.51979,54.34465,10.17514,10.17514,False,False,NSW1 higher than VIC1,9834.07,3891.103892,20.448633,20.99,25.0,15.37,False,False,True,False,Not a spike,1,Not applicable
2,2026-02-01 00:15:00,NSW1,65.08727,0,8202.70,13198.51267,-1006.50,0,4995.81267,False,0.56748,4.479816,2026-02-01,0,Sunday,True,20,9.1600,-1053.75439,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,65.08727,19.18000,45.90727,45.90727,False,False,NSW1 higher than VIC1,9834.07,3891.103892,20.448633,20.99,25.0,15.37,False,False,False,False,Not a spike,0,Not applicable
3,2026-02-01 00:20:00,NSW1,64.89000,0,8213.41,13169.71135,-1011.73,0,4956.30135,False,-0.19727,3.893369,2026-02-01,0,Sunday,True,20,4.3600,-1061.42631,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,64.89000,19.18000,45.71000,45.71000,False,False,NSW1 higher than VIC1,9834.07,3891.103892,20.448633,20.99,25.0,15.37,False,False,False,False,Not a spike,0,Not applicable
4,2026-02-01 00:25:00,NSW1,63.47180,0,8055.53,13139.84760,-934.04,0,5084.31760,False,-1.41820,3.381808,2026-02-01,0,Sunday,True,20,4.3600,-1060.65174,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,63.47180,15.18326,48.28854,48.28854,False,False,NSW1 higher than VIC1,9834.07,3891.103892,20.448633,20.99,25.0,15.37,False,False,False,False,Not a spike,0,Not applicable


## Step 5: Review Spike Classification Distribution

### What we are doing
We are checking how many spike events fall into each driver classification and confidence level.

### Why it matters
This helps us understand the dominant recommendation themes before creating decision intelligence text.


In [8]:
decision_input_summary = (
    spike_explanation
    .groupby(["regionid", "spike_driver_classification", "classification_confidence"])
    .agg(
        spike_intervals=("settlementdate", "count"),
        average_rrp=("rrp", "mean"),
        max_rrp=("rrp", "max"),
        average_absolute_price_spread=("absolute_price_spread", "mean"),
        average_congestion_signal_count=("congestion_signal_count", "mean"),
        average_active_constraints=("active_constraint_count", "mean"),
    )
    .reset_index()
    .sort_values(["regionid", "spike_intervals"], ascending=[True, False])
)

decision_input_summary


,regionid,spike_driver_classification,classification_confidence,spike_intervals,average_rrp,max_rrp,average_absolute_price_spread,average_congestion_signal_count,average_active_constraints
0,NSW1,Constraint-driven,High,27,1522.014782,9914.02372,1478.041334,1.703704,26.592593
1,NSW1,Interconnector/congestion-driven,High,26,3346.608450,20300.00000,3323.216936,2.192308,22.153846


## Step 6: Define Decision Recommendation Logic

### What we are doing
We are creating a function that maps each spike classification to a market situation, insight, recommendation, and risk.

### Why it matters
This converts analytical classification into decision-ready language for traders, analysts, and portfolio reviewers.


In [9]:
def build_decision_recommendation(row):
    classification = row["spike_driver_classification"]

    if classification == "Constraint-driven":
        return pd.Series({
            "market_situation": "Price spike occurred during elevated constraint activity.",
            "insight": (
                "Network constraint signals were present during the spike interval, "
                "suggesting dispatch limitations may have contributed to price formation."
            ),
            "recommendation": (
                "Review binding or high-impact constraints around the interval and monitor "
                "whether similar constraints recur during peak demand or outage conditions."
            ),
            "risk": (
                "Constraint-driven price outcomes can repeat quickly if the underlying "
                "network limitation remains active."
            ),
        })

    if classification == "Interconnector/congestion-driven":
        return pd.Series({
            "market_situation": "Price spike occurred during interconnector or regional separation stress.",
            "insight": (
                "Congestion-style signals or NSW1-VIC1 price divergence were present, "
                "indicating that regional transfer conditions may have influenced the price outcome."
            ),
            "recommendation": (
                "Monitor interconnector flow, flow changes, and regional price spreads. "
                "Prioritise review of NSW1-VIC1 and neighbouring interconnector behaviour."
            ),
            "risk": (
                "Regional prices may separate sharply if transfer capability is limited or flows "
                "cannot respond to price differences."
            ),
        })

    if classification == "Demand-driven":
        return pd.Series({
            "market_situation": "Price spike occurred during tight demand and supply conditions.",
            "insight": (
                "Demand or supply-demand gap signals indicate that regional market tightness "
                "may have contributed to the price spike."
            ),
            "recommendation": (
                "Monitor demand forecasts, available generation, reserve conditions, and evening "
                "peak exposure."
            ),
            "risk": (
                "High demand with limited available generation can increase exposure to scarcity "
                "pricing and fast price escalation."
            ),
        })

    if classification == "Volatility-driven":
        return pd.Series({
            "market_situation": "Price spike occurred during elevated short-term price volatility.",
            "insight": (
                "Large price changes or high rolling volatility indicate unstable price conditions "
                "around the spike interval."
            ),
            "recommendation": (
                "Review preceding dispatch intervals and monitor whether volatility persists across "
                "adjacent intervals."
            ),
            "risk": (
                "Volatile conditions can create rapid price movements that are difficult to manage "
                "without close interval-level monitoring."
            ),
        })

    return pd.Series({
        "market_situation": "Price spike occurred without a clear dominant driver.",
        "insight": (
            "Available demand, volatility, constraint, interconnector, and divergence signals "
            "do not point to one clear explanation."
        ),
        "recommendation": (
            "Review detailed market notices, constraint equations, bidding behaviour, generator "
            "availability, and interconnector limits for the interval."
        ),
        "risk": (
            "Unknown drivers create model risk and require deeper investigation before drawing "
            "commercial conclusions."
        ),
    })


## Step 7: Apply Decision Recommendation Logic

### What we are doing
We are applying the recommendation function to every classified spike event.

### Why it matters
This creates a structured decision intelligence table with business-facing interpretation for each price spike.


In [11]:
decision_fields = spike_explanation.apply(
    build_decision_recommendation,
    axis=1
)

decision_recommendations = pd.concat(
    [spike_explanation, decision_fields],
    axis=1
)

decision_recommendations.head()


,settlementdate,regionid,rrp,spike_driver_classification,classification_confidence,driver_signal_count,demand_tightness_signal,volatility_signal,constraint_signal,interconnector_congestion_signal,totaldemand,availablegeneration,supply_demand_gap,price_change,rolling_rrp_volatility_1h,active_constraint_count,max_constraint_marginal_value,congestion_signal_count,vic_nsw_congestion_signal_flag,absolute_price_spread,divergence_flag,extreme_divergence_flag,market_situation,insight,recommendation,risk
0,2026-02-05 14:30:00,NSW1,20300.00000,Interconnector/congestion-driven,High,3,True,True,False,True,10190.15,15492.52355,5302.37355,19842.65742,5793.809073,24,12.12,4.0,True,20308.00000,True,True,Price spike occurred during interconnector or ...,Congestion-style signals or NSW1-VIC1 price di...,"Monitor interconnector flow, flow changes, and...",Regional prices may separate sharply if transf...
1,2026-02-06 14:05:00,NSW1,19036.53000,Interconnector/congestion-driven,High,2,False,True,False,True,9651.93,15075.06206,5423.13206,18882.33000,5801.771238,23,5.34,2.0,False,19022.81000,True,True,Price spike occurred during interconnector or ...,Congestion-style signals or NSW1-VIC1 price di...,"Monitor interconnector flow, flow changes, and...",Regional prices may separate sharply if transf...
2,2026-02-05 18:05:00,NSW1,9914.02372,Constraint-driven,High,4,True,True,True,True,11811.76,13709.77609,1898.01609,9634.29090,3719.949900,22,20.23,2.0,False,9794.69790,True,True,Price spike occurred during elevated constrain...,Network constraint signals were present during...,Review binding or high-impact constraints arou...,Constraint-driven price outcomes can repeat qu...
3,2026-02-05 17:55:00,NSW1,9911.42681,Interconnector/congestion-driven,High,3,True,True,False,True,11901.45,14019.02209,2117.57209,9608.77893,2762.209380,24,4.96,2.0,False,9854.92681,True,True,Price spike occurred during interconnector or ...,Congestion-style signals or NSW1-VIC1 price di...,"Monitor interconnector flow, flow changes, and...",Regional prices may separate sharply if transf...
4,2026-02-05 16:35:00,NSW1,9240.83000,Constraint-driven,High,4,True,True,True,True,11896.59,14384.92803,2488.33803,8641.47000,2519.281003,29,4.34,2.0,False,9231.88000,True,True,Price spike occurred during elevated constrain...,Network constraint signals were present during...,Review binding or high-impact constraints arou...,Constraint-driven price outcomes can repeat qu...


## Step 8: Add Recommendation Confidence

### What we are doing
We are carrying forward the classification confidence and translating it into a decision confidence field.

### Why it matters
Recommendations should be interpreted with context. A high-confidence classification supports stronger monitoring conclusions than a low-confidence or unknown classification.


In [12]:
decision_recommendations["decision_confidence"] = (
    decision_recommendations["classification_confidence"]
)

decision_recommendations[
    [
        "settlementdate",
        "regionid",
        "rrp",
        "spike_driver_classification",
        "decision_confidence",
        "market_situation",
        "recommendation",
    ]
].head()


,settlementdate,regionid,rrp,spike_driver_classification,decision_confidence,market_situation,recommendation
0,2026-02-05 14:30:00,NSW1,20300.00000,Interconnector/congestion-driven,High,Price spike occurred during interconnector or ...,"Monitor interconnector flow, flow changes, and..."
1,2026-02-06 14:05:00,NSW1,19036.53000,Interconnector/congestion-driven,High,Price spike occurred during interconnector or ...,"Monitor interconnector flow, flow changes, and..."
2,2026-02-05 18:05:00,NSW1,9914.02372,Constraint-driven,High,Price spike occurred during elevated constrain...,Review binding or high-impact constraints arou...
3,2026-02-05 17:55:00,NSW1,9911.42681,Interconnector/congestion-driven,High,Price spike occurred during interconnector or ...,"Monitor interconnector flow, flow changes, and..."
4,2026-02-05 16:35:00,NSW1,9240.83000,Constraint-driven,High,Price spike occurred during elevated constrain...,Review binding or high-impact constraints arou...


## Step 9: Create Decision Recommendation Summary

### What we are doing
We are summarising recommendation themes by region, classification, and confidence.

### Why it matters
This summary is useful for Power BI and final reporting because it shows the dominant market situations and recommendation categories.


In [13]:
decision_recommendation_summary = (
    decision_recommendations
    .groupby(
        [
            "regionid",
            "spike_driver_classification",
            "decision_confidence",
            "market_situation",
        ]
    )
    .agg(
        event_count=("settlementdate", "count"),
        average_rrp=("rrp", "mean"),
        max_rrp=("rrp", "max"),
        average_absolute_price_spread=("absolute_price_spread", "mean"),
        average_congestion_signal_count=("congestion_signal_count", "mean"),
        average_active_constraints=("active_constraint_count", "mean"),
    )
    .reset_index()
    .sort_values(["regionid", "event_count"], ascending=[True, False])
)

decision_recommendation_summary


,regionid,spike_driver_classification,decision_confidence,market_situation,event_count,average_rrp,max_rrp,average_absolute_price_spread,average_congestion_signal_count,average_active_constraints
0,NSW1,Constraint-driven,High,Price spike occurred during elevated constrain...,27,1522.014782,9914.02372,1478.041334,1.703704,26.592593
1,NSW1,Interconnector/congestion-driven,High,Price spike occurred during interconnector or ...,26,3346.608450,20300.00000,3323.216936,2.192308,22.153846


## Step 10: Select Final Decision Intelligence Columns

### What we are doing
We are selecting the most important columns for the final decision recommendations output.

### Why it matters
The final CSV should be readable and dashboard-ready. It should contain the event context, classification, recommendation, risk, and confidence without unnecessary intermediate columns.


In [14]:
final_decision_recommendations = decision_recommendations[
    [
        "settlementdate",
        "regionid",
        "rrp",
        "spike_driver_classification",
        "classification_confidence",
        "decision_confidence",
        "market_situation",
        "insight",
        "recommendation",
        "risk",
        "totaldemand",
        "availablegeneration",
        "supply_demand_gap",
        "price_change",
        "rolling_rrp_volatility_1h",
        "active_constraint_count",
        "max_constraint_marginal_value",
        "congestion_signal_count",
        "vic_nsw_congestion_signal_flag",
        "absolute_price_spread",
        "divergence_flag",
        "extreme_divergence_flag",
    ]
].copy()

final_decision_recommendations.head()


,settlementdate,regionid,rrp,spike_driver_classification,classification_confidence,decision_confidence,market_situation,insight,recommendation,risk,totaldemand,availablegeneration,supply_demand_gap,price_change,rolling_rrp_volatility_1h,active_constraint_count,max_constraint_marginal_value,congestion_signal_count,vic_nsw_congestion_signal_flag,absolute_price_spread,divergence_flag,extreme_divergence_flag
0,2026-02-05 14:30:00,NSW1,20300.00000,Interconnector/congestion-driven,High,High,Price spike occurred during interconnector or ...,Congestion-style signals or NSW1-VIC1 price di...,"Monitor interconnector flow, flow changes, and...",Regional prices may separate sharply if transf...,10190.15,15492.52355,5302.37355,19842.65742,5793.809073,24,12.12,4.0,True,20308.00000,True,True
1,2026-02-06 14:05:00,NSW1,19036.53000,Interconnector/congestion-driven,High,High,Price spike occurred during interconnector or ...,Congestion-style signals or NSW1-VIC1 price di...,"Monitor interconnector flow, flow changes, and...",Regional prices may separate sharply if transf...,9651.93,15075.06206,5423.13206,18882.33000,5801.771238,23,5.34,2.0,False,19022.81000,True,True
2,2026-02-05 18:05:00,NSW1,9914.02372,Constraint-driven,High,High,Price spike occurred during elevated constrain...,Network constraint signals were present during...,Review binding or high-impact constraints arou...,Constraint-driven price outcomes can repeat qu...,11811.76,13709.77609,1898.01609,9634.29090,3719.949900,22,20.23,2.0,False,9794.69790,True,True
3,2026-02-05 17:55:00,NSW1,9911.42681,Interconnector/congestion-driven,High,High,Price spike occurred during interconnector or ...,Congestion-style signals or NSW1-VIC1 price di...,"Monitor interconnector flow, flow changes, and...",Regional prices may separate sharply if transf...,11901.45,14019.02209,2117.57209,9608.77893,2762.209380,24,4.96,2.0,False,9854.92681,True,True
4,2026-02-05 16:35:00,NSW1,9240.83000,Constraint-driven,High,High,Price spike occurred during elevated constrain...,Network constraint signals were present during...,Review binding or high-impact constraints arou...,Constraint-driven price outcomes can repeat qu...,11896.59,14384.92803,2488.33803,8641.47000,2519.281003,29,4.34,2.0,False,9231.88000,True,True


In [15]:
decision_recommendations_output = OUTPUT_DIR / "07_network_decision_recommendations.csv"
decision_summary_output = OUTPUT_DIR / "07_decision_recommendation_summary.csv"

final_decision_recommendations.to_csv(decision_recommendations_output, index=False)
decision_recommendation_summary.to_csv(decision_summary_output, index=False)

print("Saved:", decision_recommendations_output)
print("Saved:", decision_summary_output)


Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/07_network_decision_recommendations.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/07_decision_recommendation_summary.csv


## Notebook 07 Analyst Note

Notebook 07 translated classified price spike events into decision intelligence recommendations.

The notebook used the spike classifications from Notebook 06 and mapped each event to a market situation, analyst insight, recommendation, risk, and confidence level. This creates a business-facing layer that is suitable for Power BI, reporting, and portfolio presentation.

The recommendations are not trading instructions. They are decision-support outputs that help analysts and traders prioritise which intervals, constraints, interconnector conditions, and regional price separation events require closer review.

The most important output is:

`outputs/07_network_decision_recommendations.csv`

This file contains the final event-level recommendation table.


## Notebook 07 Summary Table

| Step | What We Did | Why It Matters | Output / Feature Created |
|---|---|---|---|
| Step 1-2 | Loaded libraries and defined project paths. | Prepared the notebook to use outputs from Notebook 06. | Project settings |
| Step 3 | Loaded spike explanation data. | Brought in classified spike events. | `spike_explanation` |
| Step 4 | Loaded full market classification data. | Provided wider interval-level context for decision intelligence. | `market_classification` |
| Step 5 | Reviewed classification distribution. | Identified dominant spike drivers and confidence levels. | `decision_input_summary` |
| Step 6 | Defined recommendation logic. | Mapped each spike classification to market situation, insight, recommendation, and risk. | `build_decision_recommendation()` |
| Step 7 | Applied recommendation logic to spike events. | Created event-level decision intelligence records. | `decision_recommendations` |
| Step 8 | Added decision confidence. | Carried forward classification confidence into the recommendation layer. | `decision_confidence` |
| Step 9 | Created recommendation summary. | Produced a dashboard-ready summary of recommendation themes. | `decision_recommendation_summary` |
| Step 10 | Selected final output columns. | Created a clean final recommendation table for Power BI and reporting. | `final_decision_recommendations` |
| Step 11 | Saved outputs. | Exported decision intelligence CSVs for portfolio deliverables. | `07_network_decision_recommendations.csv`, `07_decision_recommendation_summary.csv` |

### Key Analyst Takeaway

Notebook 07 turns technical classification into decision-ready market intelligence.

The project can now explain not only when price spikes occurred, but what market condition was likely involved, what an analyst should monitor, what risk is present, and how confident the recommendation is.

This is the main business-value layer of the project.
